In [ ]:
# ===========================================================================
# The head-to-head. Three arms, matched on sentence count where the labelling
# arm sets the ceiling: it is bounded by real corpus size, generation isn't.
#
# Fill in DATASET_ID_V18_HIGH from `build`'s printout before running.
# ===========================================================================
import glob
import json
import os

import pandas as pd

from ddi import label_real
from ddi.data import build_human
from ddi.experiment import log_run
from ddi.manifest import load_dataset
from ddi.mixing import subsample_by_sentence
from ddi.train import train_and_eval

DATASET_ID_V18_HIGH = "20260830-044918-c8b504"
LABEL_VERDICTS = "datasets/instances/raw/label-high-c8.jsonl"

BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}

train, dev, val = build_human()


In [ ]:

SEEDS = [0, 1, 2]

In [ ]:

# ---------------------------------------------------------------------------
# Build the three arms.
# ---------------------------------------------------------------------------
label_inst, dropped = label_real.to_instances(train, LABEL_VERDICTS)
n_label_sent = len({r["sent_id"] for r in label_inst})
print(f"labelling arm: {len(label_inst)} instances, {n_label_sent} sentences, "
      f"{dropped} dropped for shape mismatch")

v18_inst, _ = load_dataset(DATASET_ID_V18_HIGH)
n_v18_sent = len({r["sent_id"] for r in v18_inst})
print(f"generation arm (full): {len(v18_inst)} instances, {n_v18_sent} sentences")

v18_matched = subsample_by_sentence(v18_inst, n_label_sent, seed=0)
print(f"generation arm (matched to labelling): "
      f"{len(v18_matched)} instances, {n_label_sent} sentences")

arms = {
    "label-high-c8": label_inst,
    "v18-high-matched": v18_matched,
    "v18-high-full": v18_inst,
}
for name, recs in arms.items():
    pos = sum(1 for r in recs if r["label"] != "NONE") / len(recs)
    print(f"  {name:20} {len(recs):>7} instances  pos rate {pos:.3f}")


In [ ]:
import ipywidgets
# ---------------------------------------------------------------------------
# Resume support: skip (arm, seed) pairs already in the run log.
# ---------------------------------------------------------------------------
def load_done(arm_names, runs_dir="runs"):
    done = {}
    for path in sorted(glob.glob(os.path.join(runs_dir, "*.json"))):
        try:
            r = json.load(open(path))
        except Exception:
            continue
        if isinstance(r, list):
            continue
        cfg, m = r.get("config", {}), r.get("metrics", {})
        key = (cfg.get("dataset"), cfg.get("seed"))
        if key[0] in arm_names and key[1] is not None and "micro_f1_pos" in m:
            done[key] = {"arm": key[0], "seed": key[1], "f1": m["micro_f1_pos"],
                        "p": m["micro_p_pos"], "r": m["micro_r_pos"]}
    return done


done = load_done(set(arms))
print(f"\n{len(done)} runs already logged, skipping those")
rows = list(done.values())

for seed in SEEDS:
    for name, recs in arms.items():
        if (name, seed) in done:
            continue
        cfg = {**BASE, "seed": seed, "dataset": name}
        m = train_and_eval(cfg, recs, dev)
        log_run(cfg, m, notes=f"generation vs labelling head-to-head, {name}")
        rows.append({"arm": name, "seed": seed, "f1": m["micro_f1_pos"],
                    "p": m["micro_p_pos"], "r": m["micro_r_pos"]})
        print(f"{name} seed={seed} f1={m['micro_f1_pos']:.3f} "
              f"p={m['micro_p_pos']:.3f} r={m['micro_r_pos']:.3f}")

    print(f"\n===== after seed {seed} =====")
    df = pd.DataFrame(rows)
    print(df.groupby("arm")[["f1", "p", "r"]].agg(["mean", "std", "count"])
          .sort_values(("f1", "mean"), ascending=False).round(3).to_string())
    print()

print("""
reference:
  human filtered      0.790   P 0.754  R 0.829
  v18 full (low eff)  0.486   P 0.398  R 0.628   sd 0.020, n=5
  LLM annotator agreement with gold (not trained, from `agree`):  ~0.73

If v18-high-matched beats label-high-c8: generation wins even bounded to the same
sentence count the corpus can supply for labelling, which is the strong result.

If label-high-c8 wins or ties: the annotator's 0.73 agreement number did not survive
being turned into training data as well as generation did -- a real and useful
negative, and worth understanding why (label noise concentrated on which classes?
check per-class in the agree report).

v18-high-full vs v18-high-matched isolates how much generation gains purely from
not being corpus-bounded, holding the generator itself fixed.
""")